# Wildfires in Europe — IRISCC VRE

<p style="margin-top: 10px;">
  <img src="https://tools.actris-ares.eu/wp-content/uploads/2025/07/Actris-150x150.png" style="height: 78px; vertical-align: middle; margin-right: 46px;">
  <img src="https://wp1.aeris-data.fr/wp-content-aeris/uploads/sites/66/2020/06/Logo_Color.png" style="height: 52px; vertical-align: middle; margin-right: 46px;">
  <img src="https://www.icos-cp.eu/sites/default/files/2017-11/ICOS_CP_logo.png" style="height: 52px; vertical-align: middle;">
</p>

## Overview

This IRISCC VRE provides tools for accessing and visualizing data about forest fires from ACTRIS, IAGOS and ICOS. Model forecasts and back-trajectory analyses are also available as ancillary tools for improving our knowledge of the impacts of wildfire emissions.

This notebook summarises which tools are available in the VRE and how to use them. The examples refer to the Canadian fire emissions observed at the research infrastructures in summer 2025. A description of how to create your own space for new analyses is also provided.

Each tool comes from the research infrastructure that produces the data, and was written by its
own team — so the notebooks differ in scope, in the questions they answer, and in the way they
were built. As part of **Demonstrator 6 of IRISCC**, they have been harmonised as far as that made
sense: the same way of choosing what to look at, of installing what is missing, of keeping the
results, and the same wording for the parts that are common. The scientific content, and the
credit for it, remain with the original authors, named in each section below.

The aim is that someone who is not a Python or VRE specialist can go through any of them without
having to edit code.

**Contents**

1. [How to use the VRE](#howto)
2. [License](#license)
3. [The 2025 Canadian wildfire season](#context)
4. [Registration needed for specific tools](#registration)
5. [ACTRIS data exploitation tools](#actris)
6. [IAGOS data exploitation tools](#iagos)
7. [ICOS data exploitation tools](#icos)
8. [FLEXPART trajectories](#flexpart)

Let's get started!

<a id="howto"></a>
## 1. How to use the VRE

### 1.1 Work in your home folder, not in the shared workspace

The tools listed below live in the **shared workspace**, at
`/workspace/VREFolders/Wildfires_in-Europe/`. That is the right place to distribute them, but not to
work in:

* it is a shared network filesystem, noticeably slower than your home directory;
* it is **read-only**, so notebooks that download data or save figures cannot run there;

It is read-only on purpose: it holds one known version of every tool, the one that was tested, so
that nobody has to wonder which copy is current. If you improve a notebook or produce results worth
adding to the VRE, send them to **services@actris-ares.eu** — they are tested and then published.

Your home directory (`/home/jovyan`, shown as `/` at the top of the JupyterLab file browser) is a
persistent volume of your own. **Copy the VRE into your home directory and work on that copy.**

### 1.2 Copying the VRE

Copy the whole thing in one go: the tools are meant to be used together, and with the complete set
this page works from your copy too, since its links are relative.

**⬇ The simplest way is the cell below: click on it, then press ▶**

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Copy the VRE into your home directory, without using the terminal.
# The terminal commands below do exactly the same thing, if you prefer them.

import shutil
from pathlib import Path

SOURCE = Path("/workspace/VREFolders/Wildfires_in-Europe")
TARGET = Path.home() / "Wildfires_in-Europe"

# Not worth copying: produced by previous runs, or regenerated on demand. The notebooks recreate
# what they need, so leaving these behind saves a few hundred megabytes.
SKIP_DIRS = {"outputs", "exports", "data", "downloads", "__pycache__", ".ipynb_checkpoints"}
SKIP_FILES = set()          # nothing to skip by name at the moment

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
except ImportError:
    widgets = None
    print("ipywidgets is not available here - use the terminal commands below instead.")


def _wanted(rel, name):
    return not (any(part in SKIP_DIRS for part in rel.parts) or name in SKIP_FILES)


def _copy(overwrite, progress=None, label=None):
    """Copy the shared folder file by file, reporting progress as it goes.

    Reading from the shared workspace costs about a second per file whatever its size, so a
    plain copy sits silent for a minute. The progress bar is what stops people from pressing
    the button again or interrupting the kernel halfway through.
    """
    items = [p for p in SOURCE.rglob("*") if _wanted(p.relative_to(SOURCE), p.name)]
    files = [p for p in items if p.is_file()]
    if progress is not None:
        progress.max = len(files)
        progress.value = 0

    copied = skipped = 0
    total = 0
    for src in items:
        rel = src.relative_to(SOURCE)
        dst = TARGET / rel
        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
            dst.chmod(dst.stat().st_mode | 0o700)
            continue
        if dst.exists() and not overwrite:
            skipped += 1
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            # copy2 preserves the source permissions, and files in the shared workspace are
            # read-only for everyone: without this the copy in your own home
            # would still open as "notebook is read-only".
            dst.chmod(dst.stat().st_mode | 0o600)
            copied += 1
            total += src.stat().st_size
        if progress is not None:
            progress.value += 1
            if label is not None:
                label.value = f"{progress.value}/{len(files)} — {rel}"
    if label is not None:
        label.value = ""
    return copied, skipped, total


if widgets is not None:
    mode_dd = widgets.Dropdown(
        options=[("Keep the files I already have", False),
                 ("Overwrite my files with the shared version", True)],
        value=False, description="If a file already exists:",
        style={"description_width": "initial"}, layout=widgets.Layout(width="520px"))
    copy_btn = widgets.Button(description="Copy the VRE to my home folder",
                              button_style="success", icon="copy")
    progress = widgets.IntProgress(value=0, min=0, max=1, description="Copying:",
                                   style={"description_width": "initial"},
                                   layout=widgets.Layout(width="520px", visibility="hidden"))
    progress_label = widgets.HTML("")
    copy_out = widgets.Output()

    def _on_copy(_):
        with copy_out:
            copy_out.clear_output()
            if not SOURCE.exists():
                print(f"The shared folder was not found at {SOURCE}.")
                return
            existed = TARGET.exists()
            print(f"Copying {SOURCE}\n     to {TARGET}")
            print("Reading from the shared folder is slow - this takes about a minute.")
        copy_btn.disabled = True
        copy_btn.description = "Copying ..."
        progress.layout.visibility = "visible"
        try:
            copied, skipped, total = _copy(mode_dd.value, progress, progress_label)
        except Exception as exc:  # noqa: BLE001 - the reason must be readable
            with copy_out:
                print(f"Copy failed ({type(exc).__name__}: {exc})")
            return
        finally:
            copy_btn.disabled = False
            copy_btn.description = "Copy the VRE to my home folder"
            progress.layout.visibility = "hidden"
        with copy_out:
            print(f"{copied} file(s) copied, {total / 1e6:.1f} MB"
                  + (f", {skipped} left untouched" if skipped else ""))
            if existed and skipped and not mode_dd.value:
                print("Files you already had were kept. Choose the other option to replace them "
                      "with the shared version.")
            # Opening /lab/tree/<path> both opens the notebook and moves the file browser to
            # its folder, so the user does not have to navigate there by hand - and from that
            # copy the buttons below point at their own notebooks, not the shared ones.
            try:
                rel = TARGET.relative_to(Path.home())
            except ValueError:
                rel = None
            if rel is None:
                print(f"\nOpen {TARGET.name}/ in the file browser on the left and work from there.")
            else:
                prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
                url = f"{prefix}/lab/tree/{rel}/IRISCC_home.ipynb"
                display(HTML(
                    f'<p style="margin:14px 0 4px;font-weight:bold">Next step:</p>'
                    f'<p style="margin-top:0"><a href="{url}" target="_blank" '
                    f'style="padding:10px 16px;background:#1B4053;color:white;'
                    f'text-decoration:none;border-radius:5px;display:inline-block">'
                    f'Open my copy of this page</a></p>'
                    f'<p style="font-size:90%;color:#555">This opens your copy in a new tab and '
                    f'moves the file browser to <code>{rel}/</code>. From there the buttons on '
                    f'this page open <b>your</b> notebooks instead of the shared, read-only ones. '
                    f'You can then close this tab. If the link does not open, use the file browser '
                    f'on the left: go to the top level with the folder icon, then into '
                    f'<code>{rel}/</code> and open <code>IRISCC_home.ipynb</code>.</p>'))

    copy_btn.on_click(_on_copy)
    display(widgets.VBox([mode_dd, copy_btn, progress, progress_label, copy_out]))

    # Which copy of this page is running is the thing users are least sure about, and the file
    # browser on the left does not always say. Ask the kernel instead: it cannot be wrong.
    here = Path.cwd()
    if str(here).startswith("/workspace"):
        print(f"You are running this page from the SHARED workspace:\n   {here}")
        print("Press the button to copy the VRE into your home, then use the link that appears.")
    else:
        print(f"You are running this page from your own copy:\n   {here}")
        print("The buttons below open your own notebooks, which you can run and save.")
        print("Pressing the button again would refresh this copy from the shared workspace.")

**What the button just did:**

1. it copies everything into `~/Wildfires_in-Europe` — about a minute, with a progress bar;
2. it gives you a **link to your own copy of this page**;
3. clicking that link opens your copy in a new tab **and moves the file browser to it**, so there is
   nothing to navigate by hand.

Step 3 is the one that matters: from your own copy, the buttons further down open **your**
notebooks, which you can run and save. Opened from here they lead to the shared files, which are
read-only for everyone. Once your copy is open you can close this tab.

**Which copy am I in?** The cell above answers it: before the button, it prints the folder
the page is actually running from. A path under `/workspace/` is the shared, read-only one; anything
under `/home/jovyan/` is yours. The browser address bar says the same thing, and even more plainly: opened through the portal the
address is `iriscc.d4science.org/group/…`, while your own copy is on
`jupyterhub.d4science.org/user/<your username>/…`. If you can see your username in the address,
you are in your own copy.

**And the file browser on the left?** The link opens your copy in a new tab and that tab's file
browser starts in your folder. The tab you came from keeps showing the shared workspace, which is
why closing it avoids confusion. If a tab ever shows the wrong folder, the folder icon at the top of
the browser panel goes back to the top level, and from there `Wildfires_in-Europe/` is your copy.

> **If the button does not appear, or nothing happens when you press it.** The button is an
> interactive widget: part of it lives in the kernel, part in this page, and the two can get out of
> step — after running the cell a second time, or when the page has been open for a while, or when
> the notebook was saved with its outputs. Use *Edit → Clear Outputs of All Cells*, run the cell
> again, and if it is still unresponsive reload the browser page. Nothing is broken and nothing is
> lost: the copy can be repeated safely, since files you already have are kept.

> **What happens to files you already have.** By default they are **kept**, so pressing the button
> a second time cannot overwrite work you have done — it only fills in what is missing. The other
> side of that is that you will not receive updates either: if the shared notebooks have been
> improved and you want the new version, choose **"Overwrite my files with the shared version"** in
> the dropdown. That does replace your edits, so copy anything you want to keep somewhere else
> first — a notebook you have modified is not recoverable once overwritten.

Data produced by previous runs is skipped (`data/`, `downloads/`, `outputs/`, `exports/`).

**If you prefer the terminal**, this does the same thing (*File → New → Terminal*):

```bash
cp -r /workspace/VREFolders/Wildfires_in-Europe ~/
chmod -R u+rwX ~/Wildfires_in-Europe
cd ~/Wildfires_in-Europe
```

The `chmod` matters: `cp` keeps the permissions of the shared files, which are read-only for
everyone, so without it your own copy would still open as "notebook is read-only".

Then reopen this page from **your** copy — in the file browser it is under `Wildfires_in-Europe/`,
not under `workspace/` — and use the buttons from there.

**If you only need a single tool**, copy just its folder instead — but take the **whole folder**,
not only the `.ipynb`: most of these notebooks import helper modules or read data files that sit
next to them, and copying the notebook alone gives a `ModuleNotFoundError` on the first cell. The
full guide lists the individual commands.

### 1.3 Results do not survive the next run

Each tool keeps what it downloads and produces in a folder of its own, and **empties it every time
the notebook is run again** — otherwise repeated runs would slowly fill up your home directory.

Every notebook therefore ends with an **export cell**: it packs the figures and tables of the run
into a single zip and gives you a download link. Use it before starting a new run on anything you
want to keep. A few notebooks write nothing at all — they read the data as they plot it — and say
so where the export cell would otherwise be.

### 1.4 Full guide

The complete instructions — what each folder contains, which files can be deleted after copying, how
to check your disk usage, what to do when the interactive controls stop responding, and how to share
your work back — are in **Working in your home folder**, next to this notebook:

* [Read it here](VRE_home_folder_instructions.md) — opens in JupyterLab
* [Download the PDF](VRE_home_folder_instructions.pdf) — to keep, print, or read offline

<a id="license"></a>
## 2. License

Datasets and documentation: **CC-BY 4.0 International**

Software: **GNU GPL v3.0**

<a id="context"></a>
## 3. The 2025 Canadian wildfire season

The 2025 Canadian wildfire season began in mid-May with over 160 fires, mainly in central-eastern regions, lasting until July when fires spread nationwide through the summer. By August, Atlantic Canada faced extreme heat and new fires.

Climate change has intensified wildfire conditions, producing hotter, drier weather and longer seasons. Some 2025 fires reignited from 2023 "zombie" fires. By mid-June, the season was on track to be the second-worst on record, surpassing 1989 by August and trailing only 2023.

Tens of thousands of people were evacuated and two civilians died. The Canadian Armed Forces and hundreds of international firefighters were deployed. Fires destroyed homes and infrastructure. Smoke caused hazardous air quality across Canada and the United States, even reaching Europe and tinting skies red-orange.

The plume from the Canadian fires was detected at ACTRIS, IAGOS and ICOS sites. Based on the fast-track analysis carried out by the three research infrastructures, tools for wildfire detection have been designed and included in this VRE. The sections below demonstrate how to search, select and visualize data from all three infrastructures and from model output. Wherever the available datasets allow it, the examples refer to the Canadian fire case.

<a id="registration"></a>
## 4. Registration needed for specific tools

To access and visualize the IAGOS and ICOS tools, you first need to complete a registration procedure. The instructions for each are given below.

### Create an IAGOS account

1. Register at [iagos.aeris-data.fr/registration](https://iagos.aeris-data.fr/registration/)
2. Once the IAGOS team has validated your request, you will be informed by email
3. On the [authentication page](https://iagos.aeris-data.fr/profile/), click "forgot password" and create your password

#### Generating authentication credentials

IAGOS services require authentication via AERIS SSO. You must generate a configuration file containing your credentials.

Open a terminal (*File → New → Terminal*) and run the credential generation script:

```bash
cd ~/iagos
conda env list
conda activate iagos-notebooks
PYTHONPATH=src python scripts/generate_credentials.py
```

The script will prompt you for your AERIS credentials and print the lines to add to `authentication.py`.

### Create an ICOS account

The ICOS notebooks authenticate through the `icoscp` Python library, which is how the Carbon Portal
records your acceptance of the data licence. You need an account at
[cpauth.icos-cp.eu](https://cpauth.icos-cp.eu/), and you must read and accept the ICOS Data Licence
in your user profile.

The library offers two ways in:

* **A Carbon Portal password** — entered once inside the notebook and stored for later sessions.
  This is the straightforward option.
* **An authentication token** — copied from the "My Account" page. A token is obtained after
  logging in with *any* supported mechanism, so it is the way in for accounts that sign in through
  ORCID or an institutional provider and have no Carbon Portal password. It expires in under 28
  hours, and since it is written into the notebook source it must never be left in a copy shared
  with other VRE users.

Both options are described in the authentication section of the ICOS notebooks.

<a id="actris"></a>
<img src="https://tools.actris-ares.eu/wp-content/uploads/2025/07/Actris-150x150.png" align="right" style="height: 96px; margin-left: 20px; margin-bottom: 8px;">

## 5. ACTRIS data exploitation tools

The ACTRIS tools available in the VRE allow visualization of key variables of interest for investigating wildfire emissions. Aerosol lidar and aerosol in-situ data are currently supported.

Four notebooks are available.

### 5.1 Explore ACTRIS Facilities

Navigate the stations for which data are available in the ACTRIS data centre, filtering by country and distinguishing ACTRIS-labelled facilities from other stations served by the data centre.

<p style="margin: 14px 0 18px">
<a href="actris/ACTRIS_explore_national_facilities.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ACTRIS Explore Facilities
</a>
</p>

*Developed by Lise Eder Murberg (NILU) & Ermann Ripepi (CNR)*

### 5.2 ACTRIS Aerosol Remote Sensing

Search data by station and date among the aerosol lidar stations served by the ACTRIS data centre. Examples of data download and plotting are provided for aerosol optical profiles and for a new advanced product for aerosol type identification. The examples refer to 5 June 2025, when particles emitted by the Canadian fires were detected over Potenza.

A menu near the top chooses what to run: the **full analysis**, optical profiles followed by aerosol typing, or **aerosol typing only**. Either way the notebook ends by putting HIRAC, MAC and HETEAC-Flex side by side on the same measurement, where the algorithms can be seen to agree or to disagree.

<p style="margin: 14px 0 18px">
<a href="actris/aerosol_remotesensing/ACTRIS_aerosol_remote_sensing.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ACTRIS Aerosol Remote Sensing
</a>
</p>

### 5.3 ACTRIS Aerosol Typing

If the question is which aerosol was in the air rather than what the lidar measured, this notebook
goes straight there. Same station and date selection, then the three classification products —
HIRAC, MAC and HETEAC-Flex — and the figure that puts them side by side. The optical profiles are
neither downloaded nor plotted, which makes it quicker and shorter to read.

<p style="margin: 14px 0 18px">
<a href="actris/aerosol_remotesensing/ACTRIS_aerosol_typing.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ACTRIS Aerosol Typing
</a>
</p>

*Developed by Claudio Dema, Lucia Mona, Ermann Ripepi (CNR)*

### 5.4 ACTRIS Aerosol In-Situ (quality-controlled)

Search and visualize fully quality-controlled aerosol in-situ data of interest for wildfire
detection. The current version covers data up to the end of March 2025; an update extending the
coverage is in preparation.

<p style="margin: 14px 0 18px">
<a href="actris/aerosol_in_situ/in_situ_L2/ACTRIS_aerosol_insitu_L2.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ACTRIS Aerosol In-Situ
</a>
</p>

*Developed by Lise Eder Murberg, Maria Leseth Føyen (NILU)*

### 5.5 ACTRIS Aerosol In-Situ NRT

Access and plot ACTRIS Near Real-Time in-situ data on a user-specified time scale, going back to
2023.

NRT data are distributed through two THREDDS catalogs, and the notebook crawls both, letting you
choose which one to plot from:

* the **ACTRIS** catalog holds only the last three months, as a rolling window;
* the **EBAS** catalog holds the historical NRT files back to 2023. Everything in the ACTRIS
  catalog should also be in EBAS, which in addition carries non-ACTRIS data.

This two-catalog arrangement is a temporary solution: once NRT data are available through the
metadata API, access will be simpler and the distinction will disappear.

<p style="margin: 14px 0 18px">
<a href="actris/aerosol_in_situ/in_situ_NRT/ACTRIS_aerosol_insitu_NRT.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ACTRIS Aerosol In-Situ NRT
</a>
</p>

*Developed by Lise Eder Murberg, Maria Leseth Føyen (NILU)*

<a id="iagos"></a>
<img src="https://wp1.aeris-data.fr/wp-content-aeris/uploads/sites/66/2020/06/Logo_Color.png" align="right" style="height: 64px; margin-left: 20px; margin-bottom: 8px;">

## 6. IAGOS data exploitation tools

This notebook provides tools to search and download IAGOS flight data focused on CO (carbon monoxide) from fire sources, using profile data (vertical measurements during ascent and descent).

The **IAGOS_flight_search_download** notebook is configured specifically for IRISCC:

- **Fixed parameters:** `originCO = "fire"` and `type = "profiles"`
- **Variable parameters:** `start`, `end`, `bbox`, `level`, `region`

You can search for flights without downloading, which returns the flight count and total data size, or download the data files as a ZIP archive, with a maximum limit of 1 GB per request. Plots of measured CO with L4 regional fire contributions help identify the geographic origin of the CO detected during the flight.

The example included in the VRE concerns anomalous CO values observed at Frankfurt airport on 26 July 2025, related to North American fires.

<p style="margin: 14px 0 18px">
<a href="iagos/IAGOS_flight_search_download.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    IAGOS
</a>
</p>

*Developed by Damien Boulanger & Valérie Thouret (CNRS)*

<a id="icos"></a>
<img src="https://www.icos-cp.eu/sites/default/files/2017-11/ICOS_CP_logo.png" align="right" style="height: 64px; margin-left: 20px; margin-bottom: 8px;">

## 7. ICOS data exploitation tools

Four notebooks give access to ICOS atmospheric data. In all of them, station, data type and the
variables to plot are chosen from dropdowns.

**Level 2 (quality-controlled)** — searches and visualizes data of interest for wildfire detection
and analysis: long-term series of CO₂ from flask samples, and comparisons between variables measured
at the same site. Several series can be plotted together, or two of them on separate axes, which
helps in identifying wildfire cases more precisely.

<p style="margin: 14px 0 18px">
<a href="icos/ICOS_ex1a_atmo_data_L2.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ICOS Level 2
</a>
</p>

**Near Real-Time** — accesses ICOS NRT atmospheric data: station, data type and measurement height
are selected interactively. One NRT file covers one day, so several dates can be selected at once
and are concatenated into a single series.

<p style="margin: 14px 0 18px">
<a href="icos/ICOS_ex1a_atmo_data_NRT.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ICOS NRT
</a>
</p>

### Comparing two species

The two notebooks above combine series of the **same** product — typically one gas at several
measurement heights. These two variants instead load **two different products** for the same
station, so that CO can be plotted against CO₂: each on its own axis, or one at a time.

They are recent additions and still carry a 0.x version number.

<p style="margin: 14px 0 18px">
<a href="icos/ICOS_ex1a_atmo_data_L2_two_species.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ICOS Level 2 - two species
</a>
</p>

<p style="margin: 14px 0 18px">
<a href="icos/ICOS_ex1a_atmo_data_NRT_two_species.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    ICOS NRT - two species
</a>
</p>

*Developed by Zheng Duan & Alex Vermeulen (Lund University)*

The Level 2 notebooks require an ICOS Carbon Portal account; see [Registration](#registration)
above. The NRT ones do not: near real time data are open.

For browsing the data outside the VRE, see the ICOS Data Portal at
[data.icos-cp.eu/portal](https://data.icos-cp.eu/portal/).

<a id="flexpart"></a>
## 8. FLEXPART trajectories

### 8.1 Historical analysis

The **FLEXPART_historical** notebook is a tool for exploring the historical presence of wildfire emissions over Europe.

Daily GFAS/CAMS global biomass-burning black carbon emission fluxes have been used to model wildfire aerosol for the last decade. FLEXPART analyses of black carbon concentrations are available over Europe as daily fields at 14 height levels on a 0.5° × 0.5° grid (1° × 1° before 2016). The tool allows investigation of the vertical cross-section of wildfire-related black carbon over Europe for the period 2008–2025.

<p style="margin: 14px 0 18px">
<a href="flexpart/FLEXPART_historical.ipynb" style="padding: 10px; background-color: #1B4053; color: white; text-decoration: none; border-radius: 5px; display: inline-block;">
    FLEXPART
</a>
</p>

*Developed by Nikolaos Evangeliou and Sabine Eckhardt (NILU)*

### 8.2 Recent cases

The historical tool now extends to 2025, so the 2025 Canadian fire case can be explored there.

For the current year, not yet covered by the historical analyses, a forward-trajectory analysis tool is available at [atmo-access.nilu.no/FWD_IRISCC_BBBC.py](https://atmo-access.nilu.no/FWD_IRISCC_BBBC.py). It provides both global and regional maps of biomass-burning black carbon emissions, and vertical profiles of biomass-burning black carbon emissions at ACTRIS aerosol lidar sites.

*Reference:* tools developed by Nikolaos Evangeliou and Sabine Eckhardt (NILU).
Historical FLEXPART black carbon data: [thredds.nilu.no/thredds/catalog/flexpart/IRISCC](https://thredds.nilu.no/thredds/catalog/flexpart/IRISCC/catalog.html)

## Citing and publishing your results

Datasets and products generated within this VRE can be published with a
persistent identifier, so that they can be cited in publications and reused
by others. If you have produced for example a derived product that you
consider of interest for the wildfire community, or you analyse new wildfire cases, you can request the
assignment of a DOI by contacting the VRE manager at <services@actris-ares.eu>, briefly
describing the dataset, the event it refers to and the data sources used.

Published datasets are linked from the Wildfire cases tool, which collects
documented wildfire events observed over Europe. Please note that data
retrieved from ACTRIS, IAGOS and ICOS remain subject to the data policy and
citation requirements of the respective Research Infrastructure, which apply
in addition to the citation of your own derived product.

---

*Landing page version 2.1 — 25 October 2026. Each notebook prints its own name, version and date
when you run its first cell, so you can always tell which one the kernel is running.*